In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# reset defalult plotting values
plt.rcParams['figure.figsize'] = (10, 5)
plt.rc('font', family='sans-serif')
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')
plt.rc('xtick',labelsize=14)
plt.rc('ytick',labelsize=14)

# ASTR 350: Astronomical Techniques
# Lecture 17


### Prof. Robert Quimby <br> March 17, 2026

&copy; 2026 Robert Quimby

## Quiz 13 review

## Weighted least-squares fitting

Some of your observations may be better (or more important) than others. Thus we sometimes perform weighting in the least-squares process as follows.

For example, if we have a set of linear equations, $y_i = mx_i + b$, and we want to give each equation a weight, $w_i$, when calculating the sum of the squares of the residuals from the model (with parameters $m$ and $b$), we can write:

$$S_w = \sum w_i[y_i - (mx_i + b)]^2 $$

By giving more weight we give to the $i^{\rm th}$ equation we can ensure that the best fit parameters $m$ and $b$ produce smaller residuals for the $i^{\rm th}$ equation possibly at the expense of increasing the (unweighted) residuals of other equations.

More generally we can write:
$$S_w = \sum w_i \left[ D(I_i) - M(I_i, \boldsymbol{\theta}) \right]^2$$

where $I_i$ is the "information" from the $i^{\rm th}$ observation, the function $D$ returns the observed value of the dependant variable, $\boldsymbol{\theta}$ is the set of model parameters, and the function $M$ models the information given a set of model parameters to produce a prediction of the dependant variable.

### Special Case

Set the weights equal to the inverse of the measurement uncertainty squared:
$$ w_i = 1 / \sigma^2$$

$$S_{1 / \sigma^2} = \sum { \left[ D(I_i) - M(I_i, \boldsymbol{\theta}) \right]^2 \over \sigma_i^2}$$

Note there is a much more common name for $S_{1 / \sigma^2}$

Chi-square value:    
    $$\chi^2 = \sum { \left[ D(I_i) - M(I_i, \boldsymbol{\theta}) \right]^2 \over \sigma_i^2}$$

### Matrix form with weights

Start with the system of equations, $Y = Xp$. If the measurement variances are arranged in a matrix, $C$, e.g.,

$$
C = 
\left[ \begin{array}{cccc}
\sigma_0^2 & 0   & \dots & 0 \\
0   & \sigma_1^2 & \dots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0   & 0   & \dots & \sigma_N^2 \\
\end{array} \right]
$$

then we can multiply both sides of our equations by the weight, $C^{-1}$:

$$C^{-1}Y = C^{-1}Xp$$

and the least-squares solution for $p$ is:

$$p = \left[ X^T C^{-1} X \right]^{-1} \left[X^T C^{-1} Y \right]$$

## $\chi^2$ (chi-squared) fitting
 - [notebook](../tutorials/chisq.ipynb)
 - [video](https://youtu.be/TSNV-4K3Fws) (27 min)

## $\chi^2$ fitting is flexible

- Simiple (linear) least-squares fitting as we have discussed previously works for any system of equations that can be expressed as $Y = Xp$. 
    - Our solution for $p$ does not work for systems of equations where the **parameters** to be fit are non-linear, but we can still minimize the $\chi^2$!
    - Least-squares does not work for models that cannot be expressed **analytically**, but we can still do $\chi^2$ fitting!

## The $\chi^2$ value relates to the likelihood of the model

The (relative) probability, ${\cal L}$, of randomly drawing a specific sample of data, $I$, given a model, $\theta$, is given by:

$${\cal L}\left(I \,\middle| \, \theta \right) \sim  e^{-\chi^2 / 2}$$


## Example: Brute-Force Linear Fit

In [ ]:
# load some data
data = np.genfromtxt('media/xy.dat', names='x,y')
yerr = 100
plt.errorbar(data['x'], data['y'], yerr=yerr, fmt='ro')
plt.grid();

In [ ]:
# calculate chi-squared values over a grid of test parameters
ntest = ???
m = np.linspace(200, 700, ntest)
b = np.linspace(-150, 150, ntest)
chisq = np.zeros( (ntest, ntest) )
for i in range(ntest):
    for j in range(ntest):
        model = m[i] * data['x'] + b[j]
        chisq[i, j] = ????

In [ ]:
# plot the 2D probability distribution
prob = ????
plt.imshow(prob, extent=[b[0], b[-1], m[0], m[-1]], aspect='auto', origin='lower', interpolation='None')
plt.xlabel('Offset ($b$)')
plt.ylabel('Slope ($m$)');
plt.grid();

### Determine the best-fit parameters

In [ ]:
# best fit values
i, j = ????
print(m[i], b[j])

## Marginalized probabilities give the uncertainty in each parameter

In [ ]:
# marginalize over the offset parameter
# to get the slope probability distribution
prob_m = ????
prob_m /= prob_m.max()
plt.plot(m, prob_m)
plt.grid();

In [ ]:
# find the best slope and its 1-sigma uncertainty
wbest = np.argmax(prob_m)
cdist = ????
cdist /= cdist[-1]

plt.plot(m, prob_m, lw=3)
plt.plot(m, cdist, c='k')
plt.axvline(m[wbest], c='r', ls='dashed')
plt.grid();

if False:
    #w68 = ????
    plt.fill_between(m[w68], prob_m[w68], color='0.8')
    mbest = m[wbest]
    mplus = m[w68].max() - mbest
    mminus = mbest - m[w68].min()
    print(f"best slope is {mbest:.0f} + {mplus:.0f} - {mminus:.0f}")

## The $\chi^2$ distribution

In [ ]:
xs = data['x']
model = mbest * xs + bbest
ys = model + ????
plt.errorbar(xs, ys, yerr=yerr, marker='o', ls='none');

# what is the chi-squared value for this set of fake data?
????

### Simulate thousands of random outcomes

In [ ]:
ntries = 10000
chisqs = np.zeros(ntries)
for i in range(ntries):
    ys = model + np.random.normal(0, yerr, size=data.size)
    chisqs[i] = np.sum((ys - model)**2 / yerr**2)
plt.hist(chisqs, bins=25);

In [ ]:
# show expected chi-square distribution
from scipy.stats import chi2

x = np.linspace(0, chisqs.max(), 100)
dof = data.size

plt.hist(chisqs, np.arange(60))
plt.plot(x, ????);

In [ ]:
x = np.linspace(0, 100, 100)
dof = ????
plt.plot(x, chi2.pdf(x, dof))
plt.xlabel('$\chi^2$ value ({} Degrees of Freedom)'.format(dof))
plt.ylabel('Probability Density Function')
plt.grid();

In [ ]:
plt.plot(x, chi2.cdf(x, dof))
plt.xlabel('$\chi^2$ value ({} Degrees of Freedom)'.format(dof))
plt.ylabel('Probability of a $\chi^2$ value less than than x')
plt.title('$\chi^2$ Cumulative Distribution Function (CDF)')
plt.grid();

In [ ]:
plt.plot(x, chi2.sf(x, dof))
plt.xlabel('$\chi^2$ value ({} Degrees of Freedom)'.format(dof))
plt.ylabel('Probability of a $\chi^2$ value greater than than x')
plt.title('$\chi^2$ Survival Function (1-CDF)')
plt.grid();